# Motive Raw CSV QC — Layers 1–2

Interactive front-end for parser validation and gap/missingness QC.

**Backend:** all computation lives in `motive_raw_qc.py`. This notebook only orchestrates steps and displays outputs for researcher validation.

In [1]:
import sys
from pathlib import Path

import IPython.display as display
import matplotlib.pyplot as plt
import pandas as pd

# Project root = parent of notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "motive_raw_qc.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import motive_raw_qc as mqc

versions = pd.DataFrame(
    {
        "package": ["motive_raw_qc", "pandas", "numpy", "xarray", "matplotlib"],
        "version": [
            mqc.__version__,
            pd.__version__,
            __import__("numpy").__version__,
            __import__("xarray").__version__,
            plt.matplotlib.__version__,
        ],
    }
)
display.display(versions)

,package,version
0,motive_raw_qc,0.2.0
1,pandas,3.0.3
2,numpy,2.4.6
3,xarray,2026.4.0
4,matplotlib,3.10.9


In [2]:
CONFIG_PATH = PROJECT_ROOT / "config.yaml"
config = mqc.load_config(CONFIG_PATH)
config["_base_dir"] = PROJECT_ROOT

import yaml

display_config = {k: v for k, v in config.items() if not k.startswith("_")}
display.display(yaml.safe_dump(display_config, sort_keys=False))

RepresenterError: ('cannot represent an object', PosixPath('/Users/drorhazan/Desktop/motive_qc'))

In [ ]:
try:
    layer1 = mqc.run_layer1_parse(config, verbose=True)
except (mqc.ConfigValidationError, mqc.MotiveCSVParseError, mqc.SchemaValidationError, mqc.QCValidationError) as exc:
    display.display(pd.DataFrame([{"error_type": type(exc).__name__, "message": str(exc)}]))
    raise

session = layer1.session
display.display(layer1.tables["session_summary"])
display.display(layer1.tables["marker_inventory"].head(20))
print(f"Layer 1 status: {layer1.status}; warnings/errors: {len(layer1.messages)}")

In [ ]:
layer1_tables = mqc.display_layer1_outputs(layer1)
for name, table in layer1_tables.items():
    print(f"\n=== {name} ===")
    display.display(table)

## Layer 1 validation checkpoint

Researcher review before continuing to Layer 2:

- [ ] Observed frame count matches Motive export
- [ ] Effective frame rate is correct
- [ ] Marker count is plausible
- [ ] Labeled vs unlabeled separation is correct
- [ ] Marker names match expected Motive labels
- [ ] No rigid-body/skeleton/quaternion columns treated as raw markers
- [ ] `raw_data_status` is acceptable

**Decision:** approved / needs correction

In [ ]:
try:
    layer2 = mqc.run_layer2_gaps(session, config, verbose=True)
    files_written = mqc.write_outputs(layer1, layer2, config, verbose=True)
except (mqc.QCValidationError,) as exc:
    display.display(pd.DataFrame([{"error_type": type(exc).__name__, "message": str(exc)}]))
    raise

print(f"Wrote {len(files_written)} files")

In [ ]:
layer2_views = mqc.display_layer2_outputs(layer2)

display.display(layer2_views["session_summary"])
display.display(layer2_views["marker_quality_summary"].head(20))
display.display(layer2_views["gap_events_top"])
display.display(layer2_views["gap_summary_by_group"])

for plot_name, plot_path in layer2_views["figures"].items():
    print(plot_name, plot_path)
    display.display(display.Image(filename=str(plot_path)))

## Layer 2 validation checkpoint

Researcher review before any Layer 3 work:

- [ ] `marker_quality_summary` marker count matches `marker_inventory`
- [ ] Gap start/end frames look correct on spot checks
- [ ] `duration_frames` is inclusive
- [ ] `duration_seconds = duration_frames / effective_frame_rate_hz`
- [ ] Gaps >= 0.2 s and >= 0.5 s match manual checks
- [ ] Labeled and unlabeled summaries are separated
- [ ] Plots are readable
- [ ] No gap filling, smoothing, or filtering was applied

**Decision:** approved / needs correction

In [ ]:
# Optional: record validation decision to docs/VALIDATION_LOG.md
log_path = mqc.write_validation_log(
    layer1,
    layer2,
    config,
    log_path="docs/VALIDATION_LOG.md",
    decision="pending",
    validated_by="",
    notes="Notebook run — update decision after manual review.",
)
print(f"Validation log written to: {log_path}")